# 05 Results, Reporting And Supervisor Materials

Scalony notebook z porownaniem architektur, synteza wynikow, przygotowaniem tabel, wykresow i materialow do raportu dla promotora.


## Source: `13_sequence_architecture_comparison.py`


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5
SEQ_LEN = 300
SEQ_CHANNELS = ["Speed", "Throttle", "Brake", "RPM", "nGear"]
TARGET_SUBSET = "distinctive_top5"
MAX_EPOCHS = 120
BATCH_SIZE = 16

tf.keras.utils.set_random_seed(RANDOM_STATE)


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def resample_sequence(values: np.ndarray, target_len: int) -> np.ndarray:
    values = pd.Series(values).astype(float).interpolate(limit_direction="both").bfill().ffill()
    values = values.to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return np.zeros(target_len, dtype=float)
    if n == 1:
        return np.repeat(values[0], target_len)
    x_old = np.linspace(0.0, 1.0, n)
    x_new = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_new, x_old, values)


def standardize_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std, mean, std


def standardize_sequence_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std, mean, std


def build_cnn_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_gru_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.GRU(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.GRU(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_lstm_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.LSTM(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_hybrid_cnn_model(seq_len: int, n_channels: int, n_tab_features: int, n_classes: int) -> tf.keras.Model:
    seq_in = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    tab_in = tf.keras.Input(shape=(n_tab_features,), name="tab_input")
    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(seq_in)
    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.MaxPooling1D(2)(x_seq)
    x_seq = tf.keras.layers.Dropout(0.2)(x_seq)
    x_seq = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.GlobalAveragePooling1D()(x_seq)
    x_tab = tf.keras.layers.Dense(32, activation="relu")(tab_in)
    x_tab = tf.keras.layers.Dropout(0.2)(x_tab)
    x = tf.keras.layers.Concatenate()([x_seq, x_tab])
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=[seq_in, tab_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def score_split(y_true, y_pred, prefix: str):
    return {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}_macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


telemetry_df = pd.read_csv(
    EXPORT_DIR / "best_laps_telemetry_merged.csv",
    usecols=["lap_key", "season", "round", "Driver", "sample_idx"] + SEQ_CHANNELS,
)
features_df = pd.read_csv(EXPORT_DIR / "lap_features_model_min_40_laps.csv")
subset_defs_df = pd.read_csv(EXPORT_DIR / "subset_candidate_definitions.csv")
feature_family_df = pd.read_csv(EXPORT_DIR / "feature_family_mapping.csv")
compact_recommendations_df = pd.read_csv(EXPORT_DIR / "compact_model_recommendations.csv")

drivers = subset_defs_df[subset_defs_df["subset_name"] == TARGET_SUBSET]["driver"].tolist()
telemetry_df = telemetry_df[telemetry_df["Driver"].isin(drivers)].copy()
features_df = features_df[features_df["Driver"].isin(drivers)].copy()
telemetry_df["session_key"] = telemetry_df["season"].astype(str) + "_" + telemetry_df["round"].astype(str)
features_df["session_key"] = features_df["season"].astype(str) + "_" + features_df["round"].astype(str)
telemetry_df["Brake"] = telemetry_df["Brake"].astype(str).str.lower().map({"true": 1.0, "false": 0.0})

feature_families = {
    family: feature_family_df[feature_family_df["family"] == family]["feature"].tolist()
    for family in feature_family_df["family"].unique()
}
base_excluded_cols = [
    "lap_key", "Driver", "session_key", "season", "round", "LapNumber", "Team",
    "throttle_min", "brake_min", "brake_median", "brake_hard_frac", "drs_toggle_count", "brake_diff_mean", "drs_active_frac",
    "LapTimeSeconds", "Sector1TimeSeconds", "Sector2TimeSeconds", "Sector3TimeSeconds", "sector_sum_seconds", "sector1_share", "sector2_share", "sector3_share",
]
all_features = [c for c in features_df.columns if c not in base_excluded_cols]
compact_configs = {
    "compact_no_context_no_phase": [c for c in all_features if c not in feature_families["context"] + feature_families["phase"]],
}
recommended_config = compact_recommendations_df[compact_recommendations_df["subset_name"] == TARGET_SUBSET]["recommended_config"].iloc[0]
tabular_feature_cols = compact_configs[recommended_config]

lap_rows = []
sequence_arrays = []
for lap_key, lap_df in telemetry_df.sort_values(["lap_key", "sample_idx"]).groupby("lap_key"):
    lap_meta = lap_df.iloc[0]
    channel_matrix = []
    for channel in SEQ_CHANNELS:
        series = pd.to_numeric(lap_df[channel], errors="coerce")
        channel_matrix.append(resample_sequence(series.values, SEQ_LEN))
    sequence_arrays.append(np.stack(channel_matrix, axis=-1))
    lap_rows.append(
        {
            "lap_key": lap_key,
            "Driver": lap_meta["Driver"],
            "season": int(lap_meta["season"]),
            "round": int(lap_meta["round"]),
            "session_key": lap_meta["session_key"],
            "raw_len": len(lap_df),
        }
    )

sequence_meta_df = pd.DataFrame(lap_rows)
meta_df = sequence_meta_df.merge(features_df[["lap_key"] + tabular_feature_cols], on="lap_key", how="inner")
sequence_lookup = {row["lap_key"]: idx for idx, row in sequence_meta_df.reset_index().iterrows()}
aligned_indices = meta_df["lap_key"].map(sequence_lookup).to_numpy()
X_seq = np.stack(sequence_arrays)[aligned_indices].astype(np.float32)
X_tab = meta_df[tabular_feature_cols].astype(float).to_numpy(dtype=np.float32)
y_labels = meta_df["Driver"].to_numpy()
groups = meta_df["session_key"].to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)

data_summary_df = pd.DataFrame(
    [
        {
            "n_laps": len(meta_df),
            "n_drivers": meta_df["Driver"].nunique(),
            "n_sessions": meta_df["session_key"].nunique(),
            "seq_len": SEQ_LEN,
            "n_seq_channels": len(SEQ_CHANNELS),
            "n_tab_features": len(tabular_feature_cols),
            "tabular_config": recommended_config,
        }
    ]
)
export_csv(data_summary_df, "sequence_architecture_data_summary")

outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
outer_splits = []
for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_seq, y, groups), start=1):
    inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + fold_idx)
    train_inner_rel, val_rel = next(inner_cv.split(X_seq[train_idx], y[train_idx], groups[train_idx]))
    outer_splits.append(
        {
            "fold": fold_idx,
            "train_idx": train_idx,
            "train_inner_idx": train_idx[train_inner_rel],
            "val_idx": train_idx[val_rel],
            "test_idx": test_idx,
        }
    )

split_summary_df = pd.DataFrame(
    [
        {
            "fold": s["fold"],
            "n_train": len(s["train_idx"]),
            "n_train_inner": len(s["train_inner_idx"]),
            "n_val": len(s["val_idx"]),
            "n_test": len(s["test_idx"]),
        }
        for s in outer_splits
    ]
)
export_csv(split_summary_df, "sequence_architecture_split_summary")

model_builders = {
    "cnn": lambda: build_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), len(label_encoder.classes_)),
    "gru": lambda: build_gru_model(SEQ_LEN, len(SEQ_CHANNELS), len(label_encoder.classes_)),
    "lstm": lambda: build_lstm_model(SEQ_LEN, len(SEQ_CHANNELS), len(label_encoder.classes_)),
    "hybrid_cnn_tabular": lambda: build_hybrid_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), X_tab.shape[1], len(label_encoder.classes_)),
}

history_rows = []
fold_rows = []
oof_frames = []

for model_name, builder in model_builders.items():
    oof_pred = np.full(len(y), -1, dtype=int)
    oof_max_proba = np.full(len(y), np.nan)
    oof_true_proba = np.full(len(y), np.nan)

    for split in outer_splits:
        fold_idx = split["fold"]
        train_idx = split["train_idx"]
        train_inner_idx = split["train_inner_idx"]
        val_idx = split["val_idx"]
        test_idx = split["test_idx"]

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(RANDOM_STATE + 100 * fold_idx + len(model_name))

        X_seq_train_inner, X_seq_val, X_seq_test, seq_mean, seq_std = standardize_sequence_by_train(
            X_seq[train_inner_idx], X_seq[val_idx], X_seq[test_idx]
        )
        X_seq_train_full = (X_seq[train_idx] - seq_mean) / seq_std
        X_tab_train_inner, X_tab_val, X_tab_test, tab_mean, tab_std = standardize_by_train(
            X_tab[train_inner_idx], X_tab[val_idx], X_tab[test_idx]
        )
        X_tab_train_full = (X_tab[train_idx] - tab_mean) / tab_std

        y_train_inner = y[train_inner_idx]
        y_val = y[val_idx]
        y_train_full = y[train_idx]
        y_test = y[test_idx]

        model = builder()
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=18, restore_best_weights=True)]

        if model_name == "hybrid_cnn_tabular":
            history = model.fit(
                {"seq_input": X_seq_train_inner, "tab_input": X_tab_train_inner},
                y_train_inner,
                validation_data=({"seq_input": X_seq_val, "tab_input": X_tab_val}, y_val),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=callbacks,
            )
            test_proba = model.predict({"seq_input": X_seq_test, "tab_input": X_tab_test}, verbose=0)
            train_proba = model.predict({"seq_input": X_seq_train_full, "tab_input": X_tab_train_full}, verbose=0)
        else:
            history = model.fit(
                X_seq_train_inner,
                y_train_inner,
                validation_data=(X_seq_val, y_val),
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=callbacks,
            )
            test_proba = model.predict(X_seq_test, verbose=0)
            train_proba = model.predict(X_seq_train_full, verbose=0)

        test_pred = test_proba.argmax(axis=1)
        train_pred = train_proba.argmax(axis=1)
        oof_pred[test_idx] = test_pred
        oof_max_proba[test_idx] = test_proba.max(axis=1)
        oof_true_proba[test_idx] = test_proba[np.arange(len(y_test)), y_test]

        row = {
            "model": model_name,
            "fold": fold_idx,
            "n_train": len(train_idx),
            "n_train_inner": len(train_inner_idx),
            "n_val": len(val_idx),
            "n_test": len(test_idx),
            "epochs_trained": len(history.history["loss"]),
            "best_val_loss": float(np.min(history.history["val_loss"])),
        }
        row.update(score_split(y_train_full, train_pred, "train"))
        row.update(score_split(y_test, test_pred, "test"))
        row["accuracy_gap"] = row["train_accuracy"] - row["test_accuracy"]
        row["macro_f1_gap"] = row["train_macro_f1"] - row["test_macro_f1"]
        fold_rows.append(row)

        for epoch_idx, (loss, val_loss, acc, val_acc) in enumerate(
            zip(history.history["loss"], history.history["val_loss"], history.history["accuracy"], history.history["val_accuracy"]),
            start=1,
        ):
            history_rows.append(
                {
                    "model": model_name,
                    "fold": fold_idx,
                    "epoch": epoch_idx,
                    "loss": float(loss),
                    "val_loss": float(val_loss),
                    "accuracy": float(acc),
                    "val_accuracy": float(val_acc),
                }
            )

    pred_labels = label_encoder.inverse_transform(oof_pred)
    oof_frames.append(
        pd.DataFrame(
            {
                "model": model_name,
                "lap_key": meta_df["lap_key"],
                "session_key": meta_df["session_key"],
                "true_driver": y_labels,
                "pred_driver": pred_labels,
                "pred_confidence": oof_max_proba,
                "true_class_probability": oof_true_proba,
                "is_correct": y_labels == pred_labels,
            }
        )
    )

fold_metrics_df = pd.DataFrame(fold_rows)
history_df = pd.DataFrame(history_rows)
oof_df = pd.concat(oof_frames, ignore_index=True)

summary_rows = []
for model_name in model_builders:
    model_fold = fold_metrics_df[fold_metrics_df["model"] == model_name]
    model_oof = oof_df[oof_df["model"] == model_name]
    summary_rows.append(
        {
            "model": model_name,
            "n_samples": len(model_oof),
            "mean_train_accuracy": model_fold["train_accuracy"].mean(),
            "mean_test_accuracy": model_fold["test_accuracy"].mean(),
            "mean_accuracy_gap": model_fold["accuracy_gap"].mean(),
            "mean_train_macro_f1": model_fold["train_macro_f1"].mean(),
            "mean_test_macro_f1": model_fold["test_macro_f1"].mean(),
            "mean_macro_f1_gap": model_fold["macro_f1_gap"].mean(),
            "oof_accuracy": accuracy_score(model_oof["true_driver"], model_oof["pred_driver"]),
            "oof_balanced_accuracy": balanced_accuracy_score(model_oof["true_driver"], model_oof["pred_driver"]),
            "oof_macro_f1": f1_score(model_oof["true_driver"], model_oof["pred_driver"], average="macro"),
        }
    )
summary_df = pd.DataFrame(summary_rows).sort_values(["oof_macro_f1", "oof_accuracy"], ascending=[False, False]).reset_index(drop=True)

per_driver_rows = []
confusion_frames = []
misclassified_frames = []
labels = list(label_encoder.classes_)
for model_name in model_builders:
    model_oof = oof_df[oof_df["model"] == model_name].copy()
    for driver in labels:
        mask_true = model_oof["true_driver"] == driver
        y_true_binary = mask_true.astype(int)
        y_pred_binary = (model_oof["pred_driver"] == driver).astype(int)
        per_driver_rows.append(
            {
                "model": model_name,
                "driver": driver,
                "n_samples": int(mask_true.sum()),
                "precision": precision_score(y_true_binary, y_pred_binary, zero_division=0),
                "recall": recall_score(y_true_binary, y_pred_binary, zero_division=0),
                "f1": f1_score(y_true_binary, y_pred_binary, zero_division=0),
                "mean_pred_confidence": model_oof.loc[mask_true, "pred_confidence"].mean(),
                "mean_true_class_probability": model_oof.loc[mask_true, "true_class_probability"].mean(),
            }
        )
    conf = confusion_matrix(model_oof["true_driver"], model_oof["pred_driver"], labels=labels)
    conf_long = pd.DataFrame(conf, index=labels, columns=labels).stack().reset_index()
    conf_long.columns = ["true_driver", "pred_driver", "count"]
    conf_long.insert(0, "model", model_name)
    confusion_frames.append(conf_long)
    misclassified = (
        model_oof[~model_oof["is_correct"]]
        .groupby(["true_driver", "pred_driver"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    misclassified.insert(0, "model", model_name)
    misclassified_frames.append(misclassified)

per_driver_df = pd.DataFrame(per_driver_rows).sort_values(["model", "recall"], ascending=[True, False]).reset_index(drop=True)
confusion_df = pd.concat(confusion_frames, ignore_index=True)
misclassified_df = pd.concat(misclassified_frames, ignore_index=True)

compact_reco = compact_recommendations_df[compact_recommendations_df["subset_name"] == TARGET_SUBSET].iloc[0]
comparison_rows = [
    {
        "model_family": "handcrafted_compact_logreg",
        "oof_accuracy": float(compact_reco["recommended_oof_accuracy"]),
        "oof_macro_f1": float(compact_reco["recommended_oof_macro_f1"]),
    }
]
for _, row in summary_df.iterrows():
    comparison_rows.append(
        {
            "model_family": row["model"],
            "oof_accuracy": row["oof_accuracy"],
            "oof_macro_f1": row["oof_macro_f1"],
        }
    )
comparison_df = pd.DataFrame(comparison_rows)
base_acc = comparison_df.loc[comparison_df["model_family"] == "handcrafted_compact_logreg", "oof_accuracy"].iloc[0]
base_f1 = comparison_df.loc[comparison_df["model_family"] == "handcrafted_compact_logreg", "oof_macro_f1"].iloc[0]
comparison_df["accuracy_vs_handcrafted"] = comparison_df["oof_accuracy"] - base_acc
comparison_df["macro_f1_vs_handcrafted"] = comparison_df["oof_macro_f1"] - base_f1

export_csv(fold_metrics_df, "sequence_architecture_fold_metrics")
export_csv(history_df, "sequence_architecture_training_history")
export_csv(oof_df, "sequence_architecture_oof_predictions")
export_csv(summary_df, "sequence_architecture_summary")
export_csv(per_driver_df, "sequence_architecture_per_driver_metrics")
export_csv(confusion_df, "sequence_architecture_confusion_matrix")
export_csv(misclassified_df, "sequence_architecture_misclassified_pairs")
export_csv(comparison_df, "sequence_architecture_vs_handcrafted")

print("Exported:")
for name in [
    "sequence_architecture_data_summary",
    "sequence_architecture_split_summary",
    "sequence_architecture_fold_metrics",
    "sequence_architecture_training_history",
    "sequence_architecture_oof_predictions",
    "sequence_architecture_summary",
    "sequence_architecture_per_driver_metrics",
    "sequence_architecture_confusion_matrix",
    "sequence_architecture_misclassified_pairs",
    "sequence_architecture_vs_handcrafted",
]:
    print("-", EXPORT_DIR / f"{name}.csv")

print("\nArchitecture summary:")
print(summary_df.to_string(index=False))
print("\nVersus handcrafted:")
print(comparison_df.to_string(index=False))


## Source: `14_final_results_synthesis.py`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

EXPORT_DIR = Path("exports")
TARGET_SUBSET = "distinctive_top5"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def safe_round(series: pd.Series, digits: int = 6) -> pd.Series:
    return series.astype(float).round(digits)


final_config_df = load_csv("final_model_config")
final_fold_df = load_csv("final_model_fold_metrics")
final_driver_df = load_csv("final_model_per_driver_metrics")
final_pairs_df = load_csv("final_model_misclassified_pairs")

arch_data_df = load_csv("sequence_architecture_data_summary")
arch_summary_df = load_csv("sequence_architecture_summary")
arch_driver_df = load_csv("sequence_architecture_per_driver_metrics")
arch_pairs_df = load_csv("sequence_architecture_misclassified_pairs")

drivers = final_config_df["drivers"].iloc[0]
tabular_config = final_config_df["config_name"].iloc[0]
seq_len = int(arch_data_df["seq_len"].iloc[0])
seq_channels = int(arch_data_df["n_seq_channels"].iloc[0])
tab_features = int(arch_data_df["n_tab_features"].iloc[0])

handcrafted_row = pd.DataFrame(
    [
        {
            "model": "handcrafted_compact_logreg",
            "model_family": "handcrafted",
            "subset_name": TARGET_SUBSET,
            "drivers": drivers,
            "feature_config": tabular_config,
            "sequence_config": np.nan,
            "oof_accuracy": final_driver_df["n_samples"].sum() and np.average(final_driver_df["recall"], weights=final_driver_df["n_samples"]),
            "oof_balanced_accuracy": final_driver_df["recall"].mean(),
            "oof_macro_f1": final_driver_df["f1"].mean(),
            "mean_train_accuracy": final_fold_df["train_accuracy"].mean(),
            "mean_test_accuracy": final_fold_df["test_accuracy"].mean(),
            "mean_accuracy_gap": final_fold_df["accuracy_gap"].mean(),
            "mean_train_macro_f1": final_fold_df["train_macro_f1"].mean(),
            "mean_test_macro_f1": final_fold_df["test_macro_f1"].mean(),
            "mean_macro_f1_gap": final_fold_df["macro_f1_gap"].mean(),
            "n_features": int(final_config_df["n_features"].iloc[0]),
            "seq_len": np.nan,
            "n_seq_channels": np.nan,
            "n_tab_features": int(final_config_df["n_features"].iloc[0]),
        }
    ]
)

seq_compare_df = arch_summary_df[arch_summary_df["model"].isin(["cnn", "hybrid_cnn_tabular"])].copy()
seq_compare_df["model_family"] = seq_compare_df["model"].map(
    {
        "cnn": "sequence",
        "hybrid_cnn_tabular": "hybrid",
    }
)
seq_compare_df["subset_name"] = TARGET_SUBSET
seq_compare_df["drivers"] = drivers
seq_compare_df["feature_config"] = np.where(
    seq_compare_df["model"] == "hybrid_cnn_tabular",
    tabular_config,
    None,
)
seq_compare_df["sequence_config"] = f"seq_len={seq_len};channels={seq_channels}"
seq_compare_df["n_features"] = np.where(
    seq_compare_df["model"] == "hybrid_cnn_tabular",
    tab_features,
    np.nan,
)
seq_compare_df["seq_len"] = seq_len
seq_compare_df["n_seq_channels"] = seq_channels
seq_compare_df["n_tab_features"] = np.where(
    seq_compare_df["model"] == "hybrid_cnn_tabular",
    tab_features,
    0,
)

model_compare_df = pd.concat([handcrafted_row, seq_compare_df], ignore_index=True, sort=False)
model_compare_df["rank_by_macro_f1"] = model_compare_df["oof_macro_f1"].rank(ascending=False, method="dense").astype(int)
best_acc = model_compare_df["oof_accuracy"].max()
best_f1 = model_compare_df["oof_macro_f1"].max()
model_compare_df["accuracy_vs_best"] = model_compare_df["oof_accuracy"] - best_acc
model_compare_df["macro_f1_vs_best"] = model_compare_df["oof_macro_f1"] - best_f1

comparison_cols = [
    "rank_by_macro_f1",
    "model",
    "model_family",
    "subset_name",
    "drivers",
    "feature_config",
    "sequence_config",
    "oof_accuracy",
    "oof_balanced_accuracy",
    "oof_macro_f1",
    "mean_train_accuracy",
    "mean_test_accuracy",
    "mean_accuracy_gap",
    "mean_train_macro_f1",
    "mean_test_macro_f1",
    "mean_macro_f1_gap",
    "n_features",
    "seq_len",
    "n_seq_channels",
    "n_tab_features",
    "accuracy_vs_best",
    "macro_f1_vs_best",
]
model_compare_df = model_compare_df[comparison_cols].sort_values(["rank_by_macro_f1", "model"]).reset_index(drop=True)
for col in [c for c in model_compare_df.columns if c not in {"rank_by_macro_f1", "model", "model_family", "subset_name", "drivers", "feature_config", "sequence_config"}]:
    model_compare_df[col] = safe_round(model_compare_df[col])
export_csv(model_compare_df, "final_results_model_comparison")

handcrafted_driver_long = final_driver_df[
    ["driver", "n_samples", "precision", "recall", "f1", "mean_pred_confidence", "mean_true_class_probability"]
].copy()
handcrafted_driver_long["model"] = "handcrafted_compact_logreg"

seq_driver_long = arch_driver_df[arch_driver_df["model"].isin(["cnn", "hybrid_cnn_tabular"])].copy()
seq_driver_long = seq_driver_long[
    ["model", "driver", "n_samples", "precision", "recall", "f1", "mean_pred_confidence", "mean_true_class_probability"]
]

driver_compare_long = pd.concat([handcrafted_driver_long, seq_driver_long], ignore_index=True)
driver_compare_long["error_count_est"] = np.rint(driver_compare_long["n_samples"] * (1.0 - driver_compare_long["recall"])).astype(int)
driver_compare_long["correct_count_est"] = driver_compare_long["n_samples"] - driver_compare_long["error_count_est"]
driver_compare_long = driver_compare_long.sort_values(["driver", "model"]).reset_index(drop=True)
for col in ["precision", "recall", "f1", "mean_pred_confidence", "mean_true_class_probability"]:
    driver_compare_long[col] = safe_round(driver_compare_long[col])
export_csv(driver_compare_long, "final_results_per_driver_comparison")

metric_cols = ["precision", "recall", "f1", "mean_pred_confidence", "mean_true_class_probability", "error_count_est", "correct_count_est"]
driver_wide = driver_compare_long.pivot(index="driver", columns="model", values=metric_cols)
driver_wide.columns = [f"{metric}_{model}" for metric, model in driver_wide.columns]
driver_wide = driver_wide.reset_index()

for model_name in ["cnn", "hybrid_cnn_tabular"]:
    for metric in ["precision", "recall", "f1", "mean_pred_confidence", "mean_true_class_probability"]:
        driver_wide[f"{metric}_gain_{model_name}_vs_handcrafted"] = (
            driver_wide[f"{metric}_{model_name}"] - driver_wide[f"{metric}_handcrafted_compact_logreg"]
        )
    driver_wide[f"errors_reduced_{model_name}_vs_handcrafted"] = (
        driver_wide["error_count_est_handcrafted_compact_logreg"] - driver_wide[f"error_count_est_{model_name}"]
    )

gain_rank_cols = [
    "driver",
    "recall_gain_cnn_vs_handcrafted",
    "f1_gain_cnn_vs_handcrafted",
    "errors_reduced_cnn_vs_handcrafted",
    "recall_gain_hybrid_cnn_tabular_vs_handcrafted",
    "f1_gain_hybrid_cnn_tabular_vs_handcrafted",
    "errors_reduced_hybrid_cnn_tabular_vs_handcrafted",
]
driver_gain_df = driver_wide.copy()
driver_gain_df = driver_gain_df.sort_values("recall_gain_hybrid_cnn_tabular_vs_handcrafted", ascending=False).reset_index(drop=True)
for col in driver_gain_df.columns:
    if col != "driver":
        driver_gain_df[col] = safe_round(driver_gain_df[col])
export_csv(driver_gain_df, "final_results_per_driver_gain_vs_handcrafted")


def normalize_pair_table(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    out = df.copy()
    out["model"] = model_name
    return out[["model", "true_driver", "pred_driver", "count"]]


pair_compare_df = pd.concat(
    [
        normalize_pair_table(final_pairs_df, "handcrafted_compact_logreg"),
        arch_pairs_df[arch_pairs_df["model"].isin(["cnn", "hybrid_cnn_tabular"])][["model", "true_driver", "pred_driver", "count"]],
    ],
    ignore_index=True,
)
pair_compare_df = pair_compare_df.sort_values(["model", "count", "true_driver", "pred_driver"], ascending=[True, False, True, True]).reset_index(drop=True)
export_csv(pair_compare_df, "final_results_confusion_pair_comparison")

pair_wide = pair_compare_df.pivot_table(
    index=["true_driver", "pred_driver"],
    columns="model",
    values="count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
for required in ["handcrafted_compact_logreg", "cnn", "hybrid_cnn_tabular"]:
    if required not in pair_wide.columns:
        pair_wide[required] = 0
pair_wide["cnn_vs_handcrafted_delta"] = pair_wide["cnn"] - pair_wide["handcrafted_compact_logreg"]
pair_wide["hybrid_vs_handcrafted_delta"] = pair_wide["hybrid_cnn_tabular"] - pair_wide["handcrafted_compact_logreg"]
pair_wide["hybrid_vs_cnn_delta"] = pair_wide["hybrid_cnn_tabular"] - pair_wide["cnn"]
pair_wide = pair_wide.sort_values(["hybrid_vs_handcrafted_delta", "cnn_vs_handcrafted_delta", "handcrafted_compact_logreg"], ascending=[True, True, False]).reset_index(drop=True)
export_csv(pair_wide, "final_results_confusion_pair_gain_vs_handcrafted")

recommendation_df = pd.DataFrame(
    [
        {
            "subset_name": TARGET_SUBSET,
            "drivers": drivers,
            "main_interpretable_baseline": "handcrafted_compact_logreg",
            "main_interpretable_baseline_config": tabular_config,
            "best_predictive_model": "hybrid_cnn_tabular",
            "best_predictive_seq_len": seq_len,
            "best_predictive_seq_channels": seq_channels,
            "best_predictive_tabular_features": tab_features,
            "best_predictive_tabular_config": tabular_config,
            "best_predictive_accuracy": model_compare_df.loc[model_compare_df["model"] == "hybrid_cnn_tabular", "oof_accuracy"].iloc[0],
            "best_predictive_macro_f1": model_compare_df.loc[model_compare_df["model"] == "hybrid_cnn_tabular", "oof_macro_f1"].iloc[0],
            "baseline_accuracy": model_compare_df.loc[model_compare_df["model"] == "handcrafted_compact_logreg", "oof_accuracy"].iloc[0],
            "baseline_macro_f1": model_compare_df.loc[model_compare_df["model"] == "handcrafted_compact_logreg", "oof_macro_f1"].iloc[0],
            "cnn_accuracy": model_compare_df.loc[model_compare_df["model"] == "cnn", "oof_accuracy"].iloc[0],
            "cnn_macro_f1": model_compare_df.loc[model_compare_df["model"] == "cnn", "oof_macro_f1"].iloc[0],
        }
    ]
)
for col in recommendation_df.columns:
    if col not in {"subset_name", "drivers", "main_interpretable_baseline", "main_interpretable_baseline_config", "best_predictive_model", "best_predictive_tabular_config"}:
        recommendation_df[col] = safe_round(recommendation_df[col])
export_csv(recommendation_df, "final_results_recommendation")

top_hybrid_gains = driver_gain_df[
    ["driver", "recall_gain_hybrid_cnn_tabular_vs_handcrafted", "f1_gain_hybrid_cnn_tabular_vs_handcrafted", "errors_reduced_hybrid_cnn_tabular_vs_handcrafted"]
].head(5)

print("Final synthesis exports created.")
print(model_compare_df[["rank_by_macro_f1", "model", "oof_accuracy", "oof_macro_f1"]].to_string(index=False))
print()
print("Top hybrid gains vs handcrafted:")
print(top_hybrid_gains.to_string(index=False))


## Source: `15_thesis_artifacts.py`


In [ ]:
from pathlib import Path

import pandas as pd

EXPORT_DIR = Path("exports")


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def round_numeric(df: pd.DataFrame, digits: int = 4) -> pd.DataFrame:
    out = df.copy()
    numeric_cols = out.select_dtypes(include=["number"]).columns
    out[numeric_cols] = out[numeric_cols].round(digits)
    return out


lap_filter_df = load_csv("lap_filter_summary")
best_lap_success_df = load_csv("best_laps_telemetry_success_summary")
baseline_best_df = load_csv("baseline_best_models")
grouped_best_df = load_csv("grouped_best_models")
subset_grouped_df = load_csv("subset_grouped_summary")
final_results_model_df = load_csv("final_results_model_comparison")
final_driver_gain_df = load_csv("final_results_per_driver_gain_vs_handcrafted")
final_confusion_gain_df = load_csv("final_results_confusion_pair_gain_vs_handcrafted")
sequence_arch_summary_df = load_csv("sequence_architecture_summary")
sequence_data_df = load_csv("sequence_architecture_data_summary")

final_subset_n = int(sequence_data_df["n_laps"].iloc[0])
final_subset_sessions = int(sequence_data_df["n_sessions"].iloc[0])
final_subset_drivers = int(sequence_data_df["n_drivers"].iloc[0])
final_seq_len = int(sequence_data_df["seq_len"].iloc[0])
final_seq_channels = int(sequence_data_df["n_seq_channels"].iloc[0])

pipeline_lookup = {
    "Raw qualifying laps": "00_raw_all_qualifying_laps",
    "Dry accurate clean laps": "06_track_status_eq_1",
    "Personal-best dry laps": "07a_personal_best_only",
}
clean_session_count = int(lap_filter_df.loc[lap_filter_df["step"] == "06_track_status_eq_1", "n_unique_sessions"].iloc[0])

pipeline_rows = []
for label, step in pipeline_lookup.items():
    row = lap_filter_df.loc[lap_filter_df["step"] == step].iloc[0]
    pipeline_rows.append(
        {
            "stage": label,
            "n_laps": int(row["n_rows"]),
            "n_drivers": int(row["n_unique_drivers"]),
            "n_sessions": int(row["n_unique_sessions"]),
        }
    )

pipeline_rows.extend(
    [
        {
            "stage": "Best laps extracted successfully",
            "n_laps": int(best_lap_success_df.loc[best_lap_success_df["status"] == "ok", "count"].iloc[0]),
            "n_drivers": 28,
            "n_sessions": clean_session_count,
        },
        {
            "stage": "Broad 14-driver modeling set",
            "n_laps": 766,
            "n_drivers": 14,
            "n_sessions": clean_session_count,
        },
        {
            "stage": "Final distinctive_top5 set",
            "n_laps": final_subset_n,
            "n_drivers": final_subset_drivers,
            "n_sessions": final_subset_sessions,
        },
    ]
)
pipeline_summary_df = pd.DataFrame(pipeline_rows)
export_csv(pipeline_summary_df, "thesis_table_pipeline_summary")

baseline_row = baseline_best_df.loc[baseline_best_df["experiment"] == "experiment_b_without_time_features"].iloc[0]
grouped_row = grouped_best_df.loc[grouped_best_df["experiment"] == "experiment_b_without_time_features"].iloc[0]
subset_row = subset_grouped_df.loc[subset_grouped_df["subset_name"] == "distinctive_top5"].iloc[0]
cnn_row = final_results_model_df.loc[final_results_model_df["model"] == "cnn"].iloc[0]
hybrid_row = final_results_model_df.loc[final_results_model_df["model"] == "hybrid_cnn_tabular"].iloc[0]

progression_df = pd.DataFrame(
    [
        {
            "experiment_stage": "Broad baseline",
            "dataset": "14 drivers, no-time handcrafted features",
            "validation": "Stratified 5-fold CV",
            "n_laps": 766,
            "n_drivers": 14,
            "n_sessions": 55,
            "model": "logistic_regression",
            "accuracy": baseline_row["oof_accuracy"],
            "macro_f1": baseline_row["oof_macro_f1"],
        },
        {
            "experiment_stage": "Grouped broad baseline",
            "dataset": "14 drivers, no-time handcrafted features",
            "validation": "StratifiedGroupKFold by session",
            "n_laps": 766,
            "n_drivers": 14,
            "n_sessions": 55,
            "model": "logistic_regression",
            "accuracy": grouped_row["oof_accuracy"],
            "macro_f1": grouped_row["oof_macro_f1"],
        },
        {
            "experiment_stage": "Focused handcrafted model",
            "dataset": "distinctive_top5, compact_no_context_no_phase",
            "validation": "StratifiedGroupKFold by session",
            "n_laps": int(subset_row["n_samples"]),
            "n_drivers": int(subset_row["n_drivers"]),
            "n_sessions": 55,
            "model": "handcrafted_compact_logreg",
            "accuracy": subset_row["oof_accuracy"],
            "macro_f1": subset_row["oof_macro_f1"],
        },
        {
            "experiment_stage": "Sequence baseline",
            "dataset": "distinctive_top5, resampled telemetry sequences",
            "validation": "StratifiedGroupKFold by session",
            "n_laps": final_subset_n,
            "n_drivers": final_subset_drivers,
            "n_sessions": final_subset_sessions,
            "model": "cnn",
            "accuracy": cnn_row["oof_accuracy"],
            "macro_f1": cnn_row["oof_macro_f1"],
        },
        {
            "experiment_stage": "Best final model",
            "dataset": "distinctive_top5, hybrid sequence + compact features",
            "validation": "StratifiedGroupKFold by session",
            "n_laps": final_subset_n,
            "n_drivers": final_subset_drivers,
            "n_sessions": final_subset_sessions,
            "model": "hybrid_cnn_tabular",
            "accuracy": hybrid_row["oof_accuracy"],
            "macro_f1": hybrid_row["oof_macro_f1"],
        },
    ]
)
export_csv(round_numeric(progression_df), "thesis_table_model_progression")

sequence_arch_table_df = sequence_arch_summary_df.copy()
sequence_arch_table_df["architecture"] = sequence_arch_table_df["model"].map(
    {
        "cnn": "1D CNN",
        "gru": "GRU",
        "lstm": "LSTM",
        "hybrid_cnn_tabular": "Hybrid CNN + tabular",
    }
)
sequence_arch_table_df["subset_name"] = "distinctive_top5"
sequence_arch_table_df["n_laps"] = final_subset_n
sequence_arch_table_df["n_sessions"] = final_subset_sessions
sequence_arch_table_df["seq_len"] = final_seq_len
sequence_arch_table_df["n_seq_channels"] = final_seq_channels
sequence_arch_table_df = sequence_arch_table_df[
    [
        "architecture",
        "subset_name",
        "n_laps",
        "n_sessions",
        "seq_len",
        "n_seq_channels",
        "oof_accuracy",
        "oof_macro_f1",
        "mean_train_accuracy",
        "mean_test_accuracy",
        "mean_accuracy_gap",
    ]
].sort_values("oof_macro_f1", ascending=False)
export_csv(round_numeric(sequence_arch_table_df), "thesis_table_sequence_architecture_comparison")

driver_gain_table_df = final_driver_gain_df[
    [
        "driver",
        "recall_handcrafted_compact_logreg",
        "recall_cnn",
        "recall_hybrid_cnn_tabular",
        "f1_handcrafted_compact_logreg",
        "f1_cnn",
        "f1_hybrid_cnn_tabular",
        "errors_reduced_cnn_vs_handcrafted",
        "errors_reduced_hybrid_cnn_tabular_vs_handcrafted",
    ]
].copy()
driver_gain_table_df = driver_gain_table_df.rename(
    columns={
        "recall_handcrafted_compact_logreg": "recall_handcrafted",
        "recall_cnn": "recall_cnn",
        "recall_hybrid_cnn_tabular": "recall_hybrid",
        "f1_handcrafted_compact_logreg": "f1_handcrafted",
        "f1_cnn": "f1_cnn",
        "f1_hybrid_cnn_tabular": "f1_hybrid",
        "errors_reduced_cnn_vs_handcrafted": "cnn_errors_reduced_vs_handcrafted",
        "errors_reduced_hybrid_cnn_tabular_vs_handcrafted": "hybrid_errors_reduced_vs_handcrafted",
    }
)
driver_gain_table_df = driver_gain_table_df.sort_values("hybrid_errors_reduced_vs_handcrafted", ascending=False)
export_csv(round_numeric(driver_gain_table_df), "thesis_table_hybrid_driver_gains")

confusion_reduction_df = final_confusion_gain_df[
    [
        "true_driver",
        "pred_driver",
        "handcrafted_compact_logreg",
        "cnn",
        "hybrid_cnn_tabular",
        "cnn_vs_handcrafted_delta",
        "hybrid_vs_handcrafted_delta",
    ]
].copy()
confusion_reduction_df = confusion_reduction_df.rename(
    columns={
        "handcrafted_compact_logreg": "handcrafted_count",
        "cnn": "cnn_count",
        "hybrid_cnn_tabular": "hybrid_count",
    }
)
confusion_reduction_df = confusion_reduction_df.sort_values(
    ["hybrid_vs_handcrafted_delta", "cnn_vs_handcrafted_delta", "handcrafted_count"],
    ascending=[True, True, False],
)
export_csv(round_numeric(confusion_reduction_df.head(12)), "thesis_table_hybrid_confusion_reductions")

print("Thesis artifact tables created:")
for name in [
    "thesis_table_pipeline_summary",
    "thesis_table_model_progression",
    "thesis_table_sequence_architecture_comparison",
    "thesis_table_hybrid_driver_gains",
    "thesis_table_hybrid_confusion_reductions",
]:
    print(f"- {EXPORT_DIR / (name + '.csv')}")


## Source: `17_supervisor_figures.py`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

EXPORT_DIR = Path("exports")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 200


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path


pipeline_df = load_csv("thesis_table_pipeline_summary")
progress_df = load_csv("thesis_table_model_progression")
arch_df = load_csv("thesis_table_sequence_architecture_comparison")
gains_df = load_csv("thesis_table_hybrid_driver_gains")
confusion_df = load_csv("thesis_table_hybrid_confusion_reductions")


# 1. Pipeline sizes
plot_df = pipeline_df.copy()
plot_df["label"] = [
    "Surowe\nokraz.",
    "Czyste suche\nokraz.",
    "Personal best",
    "Best laps\ntelemetria",
    "Zbior 14\nkierowcow",
    "Final top5",
]
plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=plot_df, x="label", y="n_laps", hue="label", palette="Blues_d", dodge=False, legend=False)
ax.set_title("Skala danych po kolejnych etapach filtrowania")
ax.set_xlabel("")
ax.set_ylabel("Liczba okrazen")
for i, v in enumerate(plot_df["n_laps"]):
    ax.text(i, v + max(plot_df["n_laps"]) * 0.015, f"{int(v)}", ha="center", va="bottom", fontsize=11)
savefig("pipeline_summary.png")


# 2. Model progression
prog_plot = progress_df.copy()
prog_plot["short_label"] = [
    "14 kier.\nCV",
    "14 kier.\nGroup CV",
    "Top5\nmanual",
    "Top5\nCNN",
    "Top5\nhybryda",
]
prog_long = prog_plot.melt(
    id_vars=["short_label"],
    value_vars=["accuracy", "macro_f1"],
    var_name="metric",
    value_name="score",
)
plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=prog_long, x="short_label", y="score", hue="metric", palette=["#1f77b4", "#ff7f0e"])
ax.set_title("Postep wynikow od baseline'u do modelu finalnego")
ax.set_xlabel("")
ax.set_ylabel("Wynik")
ax.set_ylim(0, 1.0)
ax.legend(title="")
savefig("model_progression.png")


# 3. Sequence architecture comparison
arch_plot = arch_df.copy()
arch_long = arch_plot.melt(
    id_vars=["architecture"],
    value_vars=["oof_accuracy", "oof_macro_f1"],
    var_name="metric",
    value_name="score",
)
metric_map = {"oof_accuracy": "Accuracy", "oof_macro_f1": "Macro F1"}
arch_long["metric"] = arch_long["metric"].map(metric_map)
plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=arch_long, x="architecture", y="score", hue="metric", palette=["#2a9d8f", "#e76f51"])
ax.set_title("Porownanie architektur sekwencyjnych")
ax.set_xlabel("")
ax.set_ylabel("Wynik")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=12)
ax.legend(title="")
savefig("sequence_architectures.png")


# 4. Driver recall comparison
driver_plot = gains_df.copy()
driver_long = driver_plot.melt(
    id_vars=["driver"],
    value_vars=["recall_handcrafted", "recall_cnn", "recall_hybrid"],
    var_name="model",
    value_name="recall",
)
label_map = {
    "recall_handcrafted": "Regresja logistyczna",
    "recall_cnn": "CNN",
    "recall_hybrid": "Hybryda",
}
driver_long["model"] = driver_long["model"].map(label_map)
plt.figure(figsize=(11, 5.5))
ax = sns.barplot(data=driver_long, x="driver", y="recall", hue="model", palette=["#577590", "#43aa8b", "#f3722c"])
ax.set_title("Skutecznosc rozpoznawania kierowcow w trzech glownych modelach")
ax.set_xlabel("Kierowca")
ax.set_ylabel("Odsetek poprawnie rozpoznanych okrazen")
ax.set_ylim(0, 1.05)
ax.legend(title="")
savefig("driver_recall_comparison.png")


# 5. Confusion reductions
conf_plot = confusion_df.head(8).copy()
conf_plot["pair"] = conf_plot["true_driver"] + " -> " + conf_plot["pred_driver"]
conf_plot = conf_plot.sort_values("hybrid_vs_handcrafted_delta")
plt.figure(figsize=(10.5, 5.8))
ax = sns.barplot(
    data=conf_plot,
    y="pair",
    x="hybrid_vs_handcrafted_delta",
    color="#264653",
)
ax.set_title("Zmiana najwazniejszych pomylek: hybryda vs regresja logistyczna")
ax.set_xlabel("Różnica liczby pomylek (ujemny wynik = poprawa)")
ax.set_ylabel("")
ax.axvline(0, color="black", linewidth=1)
savefig("hybrid_confusion_reduction.png")

print("Supervisor figures created:")
for name in [
    "pipeline_summary.png",
    "model_progression.png",
    "sequence_architectures.png",
    "driver_recall_comparison.png",
    "hybrid_confusion_reduction.png",
]:
    print(f"- {FIG_DIR / name}")


## Source: `36_supervisor_multisetup_figures.py`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


EXPORT_DIR = Path("exports")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 200


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path


short_per_driver = load_csv("final_model_per_driver_metrics")
short_compare = load_csv("final_results_model_comparison")
long_baseline = load_csv("long_horizon_baseline_model_summary")
long_sequence = load_csv("long_horizon_sequence_model_summary")
race_baseline = load_csv("race_2025_baseline_model_summary")
race_sequence = load_csv("race_2025_sequence_model_summary")
race_events = load_csv("race_2025_balanced_top5_clean_target_event_counts")


short_n = int(short_per_driver["n_samples"].sum())

short_handcrafted = short_compare.loc[short_compare["model_family"] == "handcrafted", "oof_macro_f1"].iloc[0]
short_cnn = short_compare.loc[short_compare["model_family"] == "sequence", "oof_macro_f1"].iloc[0]
short_hybrid = short_compare.loc[short_compare["model_family"] == "hybrid", "oof_macro_f1"].iloc[0]

long_handcrafted = long_baseline[
    (long_baseline["subset_name"] == "balanced_top5") &
    (long_baseline["experiment"] == "with_time_features") &
    (long_baseline["model"] == "logistic_regression")
]["oof_macro_f1"].iloc[0]
long_cnn = long_sequence[
    (long_sequence["subset_name"] == "balanced_top5") &
    (long_sequence["model"] == "cnn")
]["oof_macro_f1"].iloc[0]
long_hybrid = long_sequence[
    (long_sequence["subset_name"] == "balanced_top5") &
    (long_sequence["model"] == "hybrid_cnn_tabular")
]["oof_macro_f1"].iloc[0]

race_handcrafted = race_baseline[
    (race_baseline["experiment"] == "style_core_no_time_no_race_context") &
    (race_baseline["model"] == "logistic_regression")
]["oof_macro_f1"].iloc[0]
race_cnn = race_sequence[race_sequence["model"] == "cnn"]["oof_macro_f1"].iloc[0]
race_hybrid = race_sequence[race_sequence["model"] == "hybrid_cnn_tabular"]["oof_macro_f1"].iloc[0]


# 1. Setup sizes
size_df = pd.DataFrame(
    {
        "setup": [
            "Kwalifikacje\n2023-2025\nTop5",
            "Kwalifikacje\n2018-2025\nTop5",
            "Wyścigi 2025\nczyste okrążenia\nwyścigowe",
        ],
        "n_laps": [
            short_n,
            698,
            4334,
        ],
    }
)
plt.figure(figsize=(10.5, 5.5))
ax = sns.barplot(data=size_df, x="setup", y="n_laps", hue="setup", dodge=False, palette="Blues_d", legend=False)
ax.set_title("Skala trzech glownych setupow eksperymentalnych")
ax.set_xlabel("")
ax.set_ylabel("Liczba okrążeń")
for i, v in enumerate(size_df["n_laps"]):
    ax.text(i, v + max(size_df["n_laps"]) * 0.015, f"{int(v)}", ha="center", va="bottom", fontsize=11)
savefig("supervisor_setup_size_comparison.png")


# 2. Logistic regression vs CNN vs hybrid across setups
model_setup_df = pd.DataFrame(
    [
        {"setup": "Kwalifikacje 2023-2025", "model_family": "Regresja logistyczna", "macro_f1": short_handcrafted},
        {"setup": "Kwalifikacje 2023-2025", "model_family": "CNN", "macro_f1": short_cnn},
        {"setup": "Kwalifikacje 2023-2025", "model_family": "Hybryda", "macro_f1": short_hybrid},
        {"setup": "Kwalifikacje 2018-2025", "model_family": "Regresja logistyczna", "macro_f1": long_handcrafted},
        {"setup": "Kwalifikacje 2018-2025", "model_family": "CNN", "macro_f1": long_cnn},
        {"setup": "Kwalifikacje 2018-2025", "model_family": "Hybryda", "macro_f1": long_hybrid},
        {"setup": "Wyścigi 2025", "model_family": "Regresja logistyczna", "macro_f1": race_handcrafted},
        {"setup": "Wyścigi 2025", "model_family": "CNN", "macro_f1": race_cnn},
        {"setup": "Wyścigi 2025", "model_family": "Hybryda", "macro_f1": race_hybrid},
    ]
)
plt.figure(figsize=(11.5, 5.8))
ax = sns.barplot(
    data=model_setup_df,
    x="setup",
    y="macro_f1",
    hue="model_family",
    palette=["#577590", "#43aa8b", "#f3722c"],
)
ax.set_title("Porównanie wyników głównych modeli w trzech setupach")
ax.set_xlabel("")
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1.0)
ax.legend(title="")
savefig("supervisor_setup_model_comparison.png")


# 3. Architecture comparison for long-horizon vs race
arch_plot_df = pd.concat(
    [
        long_sequence[long_sequence["subset_name"] == "balanced_top5"][["model", "oof_macro_f1"]].assign(setup="Kwalifikacje 2018-2025"),
        race_sequence[["model", "oof_macro_f1"]].assign(setup="Wyścigi 2025"),
    ],
    ignore_index=True,
)
arch_plot_df["model"] = arch_plot_df["model"].map(
    {
        "cnn": "CNN",
        "hybrid_cnn_tabular": "Hybryda",
        "gru": "GRU",
        "lstm": "LSTM",
    }
)
plt.figure(figsize=(11.5, 5.8))
ax = sns.barplot(
    data=arch_plot_df,
    x="model",
    y="oof_macro_f1",
    hue="setup",
    palette=["#264653", "#e76f51"],
)
ax.set_title("Porównanie architektur sekwencyjnych")
ax.set_xlabel("")
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1.0)
ax.legend(title="")
savefig("supervisor_architecture_comparison.png")


# 4. Race event coverage
event_plot_df = race_events.sort_values("round").copy()
event_plot_df["round_label"] = event_plot_df["round"].astype(int).astype(str)
plt.figure(figsize=(12.5, 5.8))
ax = sns.barplot(data=event_plot_df, x="round_label", y="n_laps", hue="n_drivers", palette="crest", dodge=False)
ax.set_title("Liczba czystych okrążeń wyścigowych w setupie wyścigowym 2025")
ax.set_xlabel("Runda")
ax.set_ylabel("Liczba okrążeń")
ax.legend(title="Liczba kierowców")
savefig("supervisor_race_event_coverage.png")


print("Supervisor multisetup figures created:")
for name in [
    "supervisor_setup_size_comparison.png",
    "supervisor_setup_model_comparison.png",
    "supervisor_architecture_comparison.png",
    "supervisor_race_event_coverage.png",
]:
    print(f"- {FIG_DIR / name}")


## Source: `38_supervisor_data_eda_figures.py`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


EXPORT_DIR = Path("exports")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 200


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def savefig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path


short_filter = load_csv("lap_filter_summary")
long_filter = load_csv("lap_filter_summary_2018_2025_strict")
race_filter = load_csv("race_2025_filter_summary")


def prepare_filter(df: pd.DataFrame, setup_name: str) -> pd.DataFrame:
    out = df.copy()
    out["setup"] = setup_name
    out["retention_pct"] = 100.0 * out["n_rows"] / out["n_rows"].iloc[0]
    label_map = {
        "00_raw_all_qualifying_laps": "Surowe okrążenia",
        "00_raw_race_laps": "Surowe okrążenia",
        "01_lap_time_not_null": "Niepusty czas okrążenia",
        "02_dry_sessions_only": "Tylko suche sesje",
        "03_is_accurate_true": "IsAccurate == True",
        "04_not_deleted": "Brak usuniętych okrążeń",
        "05_not_fastf1_generated": "Brak okrążeń generowanych",
        "06_track_status_eq_1": "TrackStatus == 1",
        "07a_personal_best_only": "Tylko personal best",
        "07_not_pit_in_out": "Bez pit-in / pit-out",
        "08_within_2s_of_stint_median": "W granicy 2 s od mediany stintu",
    }
    out["step_label"] = out["step"].map(label_map).fillna(out["step"])
    return out


filter_plot_df = pd.concat(
    [
        prepare_filter(short_filter, "Kwalifikacje 2023-2025"),
        prepare_filter(long_filter, "Kwalifikacje 2018-2025"),
        prepare_filter(race_filter, "Wyścigi 2025"),
    ],
    ignore_index=True,
)

step_order = [
    "Surowe okrążenia",
    "Niepusty czas okrążenia",
    "Tylko suche sesje",
    "IsAccurate == True",
    "Brak usuniętych okrążeń",
    "Brak okrążeń generowanych",
    "TrackStatus == 1",
    "Tylko personal best",
    "Bez pit-in / pit-out",
    "W granicy 2 s od mediany stintu",
]
filter_plot_df["step_label"] = pd.Categorical(filter_plot_df["step_label"], categories=step_order, ordered=True)
filter_plot_df = filter_plot_df.sort_values(["setup", "step_label"])

plt.figure(figsize=(12.5, 6.2))
ax = sns.lineplot(
    data=filter_plot_df,
    x="step_label",
    y="retention_pct",
    hue="setup",
    marker="o",
    linewidth=2.5,
)
ax.set_title("Redukcja liczby okrążeń po kolejnych etapach filtrowania")
ax.set_xlabel("")
ax.set_ylabel("Pozostały odsetek danych [%]")
plt.xticks(rotation=22, ha="right")
ax.legend(title="")
savefig("supervisor_filter_funnel_comparison.png")


print("Supervisor data EDA figures created:")
print(f"- {FIG_DIR / 'supervisor_filter_funnel_comparison.png'}")
